In [ ]:
# Import required libraries for data processing and text manipulation
import pandas as pd  # Data manipulation and Excel I/O
import re             # Regular expressions for text cleaning

In [ ]:
# Define input and output file paths
# Input: Excel file containing transaction data with ledger information
# Output: Will be used for exporting various frequency analysis reports
input = "credit_txn_v5.xlsx"
output = "output_file.xlsx"

In [ ]:
# Load transaction data from Excel file into a pandas DataFrame
df = pd.read_excel(input)

In [ ]:
# Create a dictionary of frequency tables grouped by Direct Group
# For each group, count the frequency of Actual Ledger Names and sort in descending order
# This helps identify which ledgers are used most frequently within each group

tables = {
    a: (
        group["Actual Ledger Name"]
        .value_counts()                                                          # Count occurrences of each ledger name
        .reset_index(name="frequency")                                           # Convert Series to DataFrame with frequency column
        .rename(columns={"index": "Actual Ledger Name"})                         # Rename index column back to Actual Ledger Name
        .sort_values(by="frequency", ascending=False)                            # Sort by frequency in descending order (most frequent first)
        .reset_index(drop=True)                                                  # Reset index after sorting
    )
    for a, group in df.groupby("Direct Group")                                  # Iterate over each Direct Group in the data
}

In [ ]:
# Export frequency tables to Excel file with sheet names corresponding to each group
# This creates a separate sheet for each Direct Group with its associated ledger frequencies
# Handles sheet name cleanup (removes illegal Excel characters, enforces 31-char limit, manages duplicates)

output_file = "ledger_freq_group-wise_sheetname_as_group.xlsx"
used_sheet_names = set()

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for a, table in tables.items():
        # Convert group name to string and clean illegal Excel characters: \/*?:[]
        sheet_name = str(a)
        sheet_name = re.sub(r'[\\/*?:\[\]]', '', sheet_name).strip()

        # Handle empty sheet names (fallback to generic name)
        if not sheet_name:
            sheet_name = "Group"

        # Excel sheet names have a maximum length of 31 characters
        sheet_name = sheet_name[:31]

        # Ensure unique sheet names by appending numeric suffix if duplicate exists
        original = sheet_name
        i = 1
        while sheet_name in used_sheet_names:
            suffix = f"_{i}"
            sheet_name = original[:31 - len(suffix)] + suffix
            i += 1

        used_sheet_names.add(sheet_name)
        table.to_excel(writer, sheet_name=sheet_name, index=False)  # Write table to sheet without row indices

In [ ]:
# Calculate ledger frequency analysis at the company level
# Groups transactions by Company ID, Company Name, and Actual Ledger Name
# Then counts the frequency of each ledger per company
# Finally sorts by Company ID (ascending) and frequency (descending)

ledger_freq = (
    df
    .groupby(
        ['Company ID', 'Company Name', 'Actual Ledger Name'],
        as_index=False                                             # Keep groupby columns as regular columns, not index
    )
    .size()                                                         # Count the number of transactions in each group
    .rename(columns={'size': 'frequency'})                          # Rename the count column to 'frequency'
    .sort_values(
        ['Company ID', 'frequency'],
        ascending=[True, False]                                    # Sort by Company ID (asc) then frequency (desc)
    )
)

In [ ]:
# Calculate group (category) frequency analysis at the company level
# Groups transactions by Company ID, Company Name, and Group classification
# Counts the frequency of each group per company
# Sorts by Company ID and frequency for easy comparison

group_freq = (
    df
    .groupby(
        ['Company ID', 'Company Name', 'Group'],
        as_index=False                                             # Keep groupby columns as regular columns
    )
    .size()                                                         # Count transactions in each group combination
    .rename(columns={'size': 'frequency'})                          # Rename count column
    .sort_values(
        ['Company ID', 'frequency'],
        ascending=[True, False]                                    # Ascending Company ID, descending frequency
    )
)

In [ ]:
# Identify ledgers whose frequency is above the average frequency for their company
# This helps highlight which ledgers are more frequently used than the company average
# Useful for analyzing transaction concentration and ledger popularity

ledgers_above_avg = (
    ledger_freq
    .assign(
        # Calculate average frequency for each company and add it as a new column
        avg_freq=ledger_freq.groupby('Company ID')['frequency'].transform('mean')
    )
    .query('frequency > avg_freq')                                 # Filter only ledgers with above-average frequency
)

In [ ]:
# Prepare final ledger frequency data sorted by Company Name and frequency
# Select relevant columns and sort to create a clean export
# This provides an easy-to-read company-wise view of ledger usage

export_df = (
    ledger_freq[
        [
            'Company Name',
            'Company ID',
            'Actual Ledger Name',
            'frequency'
        ]
    ]
    .sort_values(
        by=['Company Name', 'frequency'],
        ascending=[True, False]                                    # Sort by company name (A-Z), then frequency (high-to-low)
    )
)

# Export to Excel for business analysis and reporting
export_df.to_excel(
    "ledger_freq_company-wise.xlsx",
    index=False                                                    # Don't include row indices in the output
)

In [ ]:
# Consolidate group-wise ledger frequency data from all groups
# Combines the individual group frequency tables created earlier into a single DataFrame
# Preserves group information and sorts by group and frequency

final_df = pd.concat(
    [
        # For each group and its frequency table, add the group name as a new column
        table.assign(group=a)
        for a, table in tables.items()
    ],
    ignore_index=True                                              # Reset index after concatenation
)

# Clean up column names and restructure for final export
final_df = (
    final_df[
        ['group', 'Actual Ledger Name', 'frequency']
    ]
    .rename(columns={'frequency': 'freq'})                         # Rename to shorter alias
    .sort_values(
        by=['group', 'freq'],
        ascending=[True, False]                                    # Sort by group name (A-Z), then frequency (high-to-low)
    )
)

# Export consolidated group-wise frequency data to Excel
final_df.to_excel(
    "ledger_freq_group-wise.xlsx",
    sheet_name="Group_Ledger_Frequency",
    index=False                                                    # Don't include row indices
)